**Remark:** In this notebook we will show how to use LLMs to annotate a drug wrt "Level 1 Reactome pathways". Since we're just using putative targets and MOA as strings we are not actually restricted to using drugs from GDSC and PRISM. I.e.: this notebook can be used to <ins>**annotate any drug**</ins>

# 0. Libraries import

Import all necessary libraries and set up the environment. This includes:
- Standard data manipulation libraries (pandas, numpy)
- File system and path management (os, sys, pathlib)
- The vLLM library for efficient large language model inference

In [2]:
# !pip install huggingface_hub

In [2]:
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from vllm import LLM, SamplingParams

/villa/rhh25/Cellhit/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-06 15:28:25,569	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


We'll also set the current working directory and define important file paths.

In [3]:
#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') #replace with yours
os.chdir(library_path)
#sys.path.append(library_path) use this in case it is still not working

We are then ready to import dedicatd fucntions from the custom library

In [4]:
from CellHit.LLMs import fetch_abstracts, generate_prompt
from CellHit.data import obtain_drugs_metadata

In [5]:
#specify the data_path
data_path = library_path/'data'
describe_prompt_path = data_path/'prompts'/'drug_description_prompt.txt'
refine_prompt_path = data_path/'prompts'/'drug_refiner_prompt.txt'
model_path = Path('~/Cellhit/mixtral/').expanduser() # note that the model needs to be downloaded from huggingface, the one used can be found at https://huggingface.co/TheBloke/Mixtral-8x7B-Instruct-v0.1-GPTQ (gptq-4bit-32g-actorder_True)

In [6]:
model_path

PosixPath('/villa/rhh25/Cellhit/mixtral')

# 1. Getting a drug abstracts

Taking advantage of the custom functions provided in cellHit we can get metadata related to the two dataset used in the manuscript (i.e.: gdsc and prism)

In [6]:
dataset = 'gdsc'
metadata = obtain_drugs_metadata(dataset,data_path)
metadata.head()

,DrugID,Drug,PUTATIVE_TARGET,MOA,DRUG_SYNONYMS,SMILES,repurposing_target
0,1003,Camptothecin,TOP1,DNA replication,"Camptothecine, (+)-Camptothecin",CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,TOP1
1,1004,Vinblastine,Microtubule destabiliser,Mitosis,Velban,CC[C@@]1(C[C@H]2C[C@@](C3=C(CCN(C2)C1)C4=CC=CC...,TUBB
2,1005,Cisplatin,DNA crosslinker,DNA replication,"cis-Diammineplatinum(II) dichloride, Platinol,...",N.N.Cl[Pt]Cl,DNA
3,1006,Cytarabine,Antimetabolite,Other,"Ara-Cytidine, Arabinosyl Cytosine, U-19920",C1=CN(C(=O)N=C1N)[C@H]2[C@H]([C@@H]([C@H](O2)C...,DNA synthesis
4,1007,Docetaxel,Microtubule stabiliser,Mitosis,"RP-56976, Taxotere",CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@...,TUBB


After having selected the drug of interest we can fix its name

In [7]:
drug_name = 'Docetaxel'

and afterwards extraxt known metadata realted to this drug such as putative targets and mechanism of action

In [8]:
metadata.loc[metadata['Drug']==drug_name]

,DrugID,Drug,PUTATIVE_TARGET,MOA,DRUG_SYNONYMS,SMILES,repurposing_target
4,1007,Docetaxel,Microtubule stabiliser,Mitosis,"RP-56976, Taxotere",CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@...,TUBB


In [9]:
mechanism_of_action = metadata.loc[metadata['Drug']==drug_name]['MOA'].values[0]
putative_target = metadata.loc[metadata['Drug']==drug_name]['repurposing_target'].values[0]
mechanism_of_action,putative_target

('Mitosis', 'TUBB')

In [10]:
drug_name

'Docetaxel'

We then use the  `fetch_abstracts` function to get the abstracts related to the selected drug

In [11]:
abstracts = fetch_abstracts(drug_name,mail=None,k=10)
abstracts = [str(a) for a in abstracts]
abstracts

/villa/rhh25/Cellhit/venv/lib/python3.10/site-packages/Bio/Entrez/__init__.py:694: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Abstract 9 is not available.



['<b>Background:</b> Prostate cancer is usually considered as immune "cold" tumor with poor immunogenic response and low density of tumor-infiltrating immune cells, highlighting the need to explore clinically actionable strategies to sensitize prostate cancer to immunotherapy. In this study, we investigated whether docetaxel-based chemohormonal therapy induces immunologic changes and potentiates checkpoint blockade immunotherapy in prostate cancer. <b>Methods:</b> We performed transcriptome and histopathology analysis to characterize the changes of prostate cancer immune microenvironment before and after docetaxel-based chemohormonal therapy. Furthermore, we investigated the therapeutic benefits and underlying mechanisms of chemohormonal therapy combined with anti-PD1 blockade using cellular experiments and xenograft prostate cancer models. Finally, we performed a retrospective cohort analysis to evaluate the antitumor efficacy of anti-PD1 blockade alone or in combination with docetaxe

# 2. Generating a drug description

We now make us of the VLLM library to do serve the LLM

In [12]:
ls

AsyncDistribJobs/      mixtral/
CellHit/               notebooks/
cellHit.yml            NOTICE
data/                  protein.physical.links.v12.0.txt.gz
drugcomb_moa_data.csv  README.md
get-pip.py             requirements.txt
hfd.sh*                scripts/
learning_workflow.png  venv/
LICENSE


In [13]:
str(model_path)

'/villa/rhh25/Cellhit/mixtral'

In [14]:
!pip install "huggingface_hub==0.20.3"

In [15]:
llm = LLM(
            model=str(model_path),
            revision="gptq-4bit-32g-actorder_True",
            quantization="gptq",
            dtype='float16',
            gpu_memory_utilization=1,
            enforce_eager=True)
            #**{"attn_implementation":"flash_attention_2"}) Turn on for faster decoding (Ampere and newer GPUs only)

#initialize sampling parameters
sampling_params = SamplingParams(temperature=0.2, top_p=0.95, max_tokens=1024)


WARNING 04-02 17:30:05 config.py:618] Casting torch.bfloat16 to torch.float16.
WARNING 04-02 17:30:05 config.py:193] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 04-02 17:30:05 llm_engine.py:87] Initializing an LLM engine with config: model='/villa/rhh25/Cellhit/mixtral', tokenizer='/villa/rhh25/Cellhit/mixtral', tokenizer_mode=auto, revision=gptq-4bit-32g-actorder_True, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=gptq, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, seed=0)


INFO 04-02 17:35:29 llm_engine.py:357] # GPU blocks: 6472, # CPU blocks: 2048


We generate the prompt using a dedicated function that takes care of inserting into a template the informations recovered about the selected drug

In [16]:
describe_prompt = generate_prompt(describe_prompt_path,**{'drug_name':drug_name, 'mechanism_of_action':mechanism_of_action, 'putative_targets':putative_target})
print(describe_prompt)

<|im_start|>system
**Task**: Utilizing the drug metadata that i provide you with, act as a medicinal chemist and expert molecular biologist to generate an in-depth description of the drug.

**Instructions**:
1. Analyze the provided drug metadata, including its name, synthetic labels of the mechanism of action, and putative targets.
2. Generate a detailed textual description of the drug's mechanism of action. This should include how the drug interacts with biological systems, its effect on specific cellular or molecular targets, and its overall impact on the disease or condition it is designed to treat.
3. If available, provide a detailed textual description of the drug's metabolism. This includes how the drug is metabolized in the body, the enzymes involved, the metabolic pathways, and any known metabolites formed.
4. Ensure that the information is presented in a clear, concise, and scientifically accurate manner, suitable for an audience with a background in medicinal chemistry and mo

We can than query the LLM and obtain its answer

In [20]:
sampling_params

SamplingParams(n=1, best_of=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.2, top_p=0.95, top_k=-1, min_p=0.0, seed=None, use_beam_search=False, length_penalty=1.0, early_stopping=False, stop=[], stop_token_ids=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=1024, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True)

In [27]:
outputs = llm.generate(describe_prompt, sampling_params)

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.70s/it]


In [28]:
drug_description = outputs[0].outputs[0].text
print(drug_description)


**Mechanism of Action Description**

Docetaxel is a semi-synthetic derivative of the natural compound paclitaxel, which is isolated from the bark of the Pacific yew tree (Taxus brevifolia). Docetaxel is a potent antineoplastic agent that exerts its effects by interfering with microtubule dynamics during mitosis, leading to cell cycle arrest and ultimately, apoptosis.

Docetaxel binds to the beta subunit of tubulin, a protein responsible for the polymerization and stabilization of microtubules. This interaction stabilizes microtubules, preventing their disassembly during mitosis. Consequently, this results in the inhibition of mitotic spindle formation, chromosomal separation, and ultimately, cell division. The accumulation of cells in the G2-M phase of the cell cycle triggers the onset of apoptosis, leading to the death of cancer cells.

**Metabolism Description**

Docetaxel undergoes extensive metabolism in the liver by the cytochrome P450 (CYP) enzyme system, primarily by CYP3A4. Th

# 3.Refine the drug description with abstracts

We now repeat the procedure above but this time integrating the infos generated at the previous step and integrating them with the abstracts

In [29]:
refine_prompt = generate_prompt(refine_prompt_path,**{'previous_output':drug_description,'formatted_abstracts':"\n".join([f"Abstract {i+1}: {abstract}" for i, abstract in enumerate(abstracts)])})
print(refine_prompt)

<|im_start|>system
**Task**: Review and integrate a drug summary using the provided PubMed abstracts related to the drug. Synthesize both sets of information, offering corrections and updates as necessary.

**Instructions**:
1. Carefully read the output from the previous instruction, which contains a detailed description of the drug's mechanism of action and metabolism.
2. Review the list of PubMed abstracts, disregarding any information that is not relevant to the drug and focusing on key information that pertains to the drug's mechanism of action, metabolism, and any other relevant pharmacological details.
3. Compare and contrast the information from the previous output and the PubMed abstracts. Identify any discrepancies or conflicts between the two sources.
4. In case of conflicting information, give precedence to the information contained in the PubMed abstracts, as they are based on recent scientific research and publications.
5. Integrate the verified information and the new ins

In [30]:
outputs = llm.generate(refine_prompt, sampling_params)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:44<00:00, 44.39s/it]


In [31]:
refined_drug_description = outputs[0].outputs[0].text
print(refined_drug_description)


**Mechanism of Action Description**

Docetaxel is a semi-synthetic derivative of paclitaxel, a compound isolated from the Pacific yew tree. It is a potent antineoplastic agent that exerts its effects by interfering with microtubule dynamics during mitosis, leading to cell cycle arrest and ultimately, apoptosis. Docetaxel binds to the beta subunit of tubulin, a protein responsible for the polymerization and stabilization of microtubules. This interaction stabilizes microtubules, preventing their disassembly during mitosis, which results in the inhibition of mitotic spindle formation, chromosomal separation, and ultimately, cell division. The accumulation of cells in the G2-M phase of the cell cycle triggers the onset of apoptosis, leading to the death of cancer cells.

In addition to its direct effects on microtubules, docetaxel has been found to activate the cGAS/STING pathway in prostate cancer, subsequently inducing IFN signaling and resulting in lymphocytes infiltration. This immun

In [32]:
#dump refined_drug_description to a file
with open('drug_description_refined.txt','w') as f:
    f.write(refined_drug_description)

# 4. Pathway selection

In this part of the notebook we serve the LLM model with a different library: [Guidance](https://github.com/guidance-ai/guidance). This will let us do constrained inference. 

*It is suggested to restart the notebook before running this new part. We are currently working to obtain a new version of the notebook to avoid switching libraries using [SGLang](https://github.com/sgl-project/sglang).*

In [ ]:
import gc
import torch
import guidance
import pandas as pd
from guidance import select,gen,models
from pathlib import Path

from CellHit.data import get_reactome_layers,get_pathway_genes
from CellHit.LLMs import generate_prompt,dictionary_maker,self_consistency

In [8]:
#specify the data_path
select_prompt_path = data_path/'prompts'/'mixtral_pathway_selector.txt'
model_path = Path('~/Cellhit/mixtral/').expanduser()

Using the `get_reactome_layers` from our codebase we can automatically obtain the Reactome pathways at a given level "of depth". For more info about this we refer to the method section of our manuscript

In [9]:
reactome_pathways = get_reactome_layers(data_path/'reactome',layer_number=1)['PathwayName'].tolist()

In [10]:
reactome_pathways[:10]

['ABC-family proteins mediated transport',
 'Abacavir ADME',
 'Activation of HOX genes during differentiation',
 'Adaptive Immune System',
 'Amyloid fiber formation',
 'Apoptosis',
 'Aquaporin-mediated transport',
 'Aspirin ADME',
 'Atorvastatin ADME',
 'Azathioprine ADME']

We read the drug description obtained in the previous steps (since we will use it now to associate to the drugs the Reactome pathways)

In [11]:
#read the refined drug description
with open('drug_description_refined.txt','r') as f:
    drug_description = f.read()

We then define the constrained generation logic that we will use in order to associate the pathways to the drug

In [12]:
@guidance
def inference(lm,prompt, pathway_number, pathways_list,temperature=0.7):

    break_line = "\n"

    lm += prompt

    choosen_pathways = []
    #controlled generation
    for i in range(pathway_number):
        feasible_pathways = [i for i in pathways_list if i not in set(choosen_pathways)]
        lm += f"\nRationale {i+1}:{gen(f'rationale_{i+1}',stop=break_line,temperature=temperature)}"
        lm += f"\nPathway {i+1}:{select(options=feasible_pathways,name=f'pathway_{i+1}')}\n"
        choosen_pathways.append(lm[f'pathway_{i+1}'])
    return lm

We then load the model

In [13]:
gpu_id = 0
lm = models.Transformers(str(model_path),**{"device_map":f"cuda:{gpu_id}","revision":"gptq-4bit-32g-actorder_True"})

/villa/rhh25/Cellhit/venv/lib/python3.10/site-packages/transformers/modeling_utils.py:4193: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(


In [15]:
pathway_number = 15
self_k = 5
temperature = 0.7

In [ ]:
prompt = generate_prompt(data_path/'prompts'/'mixtral_pathway_selector.txt',**{'drug_description':drug_description,'pathways_list':reactome_pathways,'pathway_number':pathway_number})
print(prompt)

**[INST] [Start of Drug Description]**

**Mechanism of Action Description**:
Docetaxel is a semisynthetic antineoplastic agent that exerts its effects by binding to and stabilizing microtubules, thereby inhibiting the dynamics of microtubule disassembly and promoting microtubule polymerization. This interference with microtubule dynamics prevents the formation of the mitotic spindle, which is essential for cell division. Consequently, docetaxel induces cell cycle arrest at the G2/M phase, leading to apoptotic cell death in proliferating cells. The primary target of docetaxel is the beta-tubulin subunit of microtubules, specifically the TUBB protein. Recent studies have also shown that docetaxel can activate the cGAS/STING pathway in prostate cancer, subsequently inducing IFN signaling, resulting in lymphocytes infiltration. This immune response facilitates T cell infiltration in a cGAS/STING-dependent manner, providing a combination immunotherapy strategy that would improve the clinica

We than deploy the self consistency procedure and repeat the selection procedure for `self_k` iterations. At the end of each iteration, results from the LLM are collected as a dictionary {'selected_pathway':'rationale'}

In [17]:
dict_list = []

for iter in range(self_k):
    lm = lm + inference(prompt,pathway_number=pathway_number, pathways_list=reactome_pathways,temperature=temperature)
    dict_list.append(dictionary_maker(lm))
    lm.reset()
    gc.collect()
    torch.cuda.empty_cache()

We then use a dedicated function to gather results across iterations and get how many times specific pathways were selected

In [18]:
output_dict = self_consistency(dict_list)
output_dict

{'Cell Cycle, Mitotic': {'count': 2,
  'rationales': [' Docetaxel primarily exerts its effects by binding to and stabilizing microtubules, thereby inhibiting the dynamics of microtubule disassembly and promoting microtubule polymerization. This leads to cell cycle arrest at the G2/M phase, resulting in apoptotic cell death in proliferating cells. The primary target of docetaxel is the beta-tubulin subunit of microtubules, specifically the TUBB protein.',
   " The 'Cell Cycle, Mitotic' pathway is directly related to docetaxel's primary target, the beta-tubulin subunit of microtubules. This pathway is responsible for mitotic cell division, and docetaxel's ability to bind and stabilize microtubules results in mitotic arrest, leading to apoptotic cell death in proliferating cells."]},
 'Cytokine Signaling in Immune system': {'count': 5,
  'rationales': [' Recent studies have shown that docetaxel can activate the cGAS/STING pathway in prostate cancer, subsequently inducing IFN signaling, re

In [19]:
len(output_dict)

39

We then list pathways that were selected at least twice for the given drug

In [20]:
#return only pathways with count > 2
for key in output_dict.keys():
    if output_dict[key]['count'] >= 2:
        print(key)

Cell Cycle, Mitotic
Cytokine Signaling in Immune system
Biological oxidations
Adaptive Immune System
Innate Immune System
Integrin signaling
Membrane Trafficking
Signal regulatory protein family interactions
Signaling by Receptor Tyrosine Kinases
Signaling by Non-Receptor Tyrosine Kinases
Cell Cycle Checkpoints
Cytosolic iron-sulfur cluster assembly
Ciprofloxacin ADME
Transcriptional regulation of granulopoiesis
Cell surface interactions at the vascular wall
Macroautophagy
MTOR signalling


Considered than the obtain pathways we can recover the associated genes through the `get_pathway_genes` function

In [19]:
#read the list of pathways together with reactome ids
reactome_pathways_ids = pd.read_csv(data_path/'reactome'/'pathways_l1.csv')

In [24]:
reactome_pathways_ids[reactome_pathways_ids['PathwayName']=='Cell Cycle Checkpoints']

,PathwayID,PathwayName,Layer
16,R-HSA-69620,Cell Cycle Checkpoints,1


In [14]:
get_pathway_genes('R-HSA-109581')

['TRAF2',
 'TRADD',
 'RIPK1',
 'FADD',
 'CASP8',
 'TNFRSF10B',
 'TNFRSF10A',
 'TNFSF10',
 'FASLG',
 'FAS',
 'CD14',
 'TLR4',
 'LY96',
 'TICAM2',
 'TICAM1',
 'ORF71',
 'MC159L',
 'APPL1',
 'DCC',
 'CASP9',
 'CASP3',
 'UNC5B',
 'DAPK2',
 'DAPK3',
 'DAPK1',
 'UNC5A',
 'MAGED1',
 'DIABLO',
 'BAK1',
 'BAX',
 'CYCS',
 'GSDMD',
 'GSDME',
 'APAF1',
 'CASP7',
 'XIAP',
 'CARD8',
 'APIP',
 'AVEN',
 'MAPK3',
 'MAPK1',
 'UACA',
 'CDKN2A',
 'C1QBP',
 'BID',
 'BCL2',
 'BAD',
 'PMAIP1',
 'BBC3',
 'BCL2L11',
 'BMF',
 'BCL2L1',
 'STAT3',
 'MAPK8',
 'DYNLL2',
 'PPP3CC',
 'PPP3R1',
 'YWHAQ',
 'YWHAH',
 'YWHAG',
 'YWHAB',
 'YWHAZ',
 'SFN',
 'YWHAE',
 'AKT3',
 'AKT2',
 'AKT1',
 'DYNLL1',
 'TP53',
 'E2F1',
 'TFDP2',
 'TFDP1',
 'TP63',
 'TP73',
 'TP53BP2',
 'PPP1R13B',
 'NMT1',
 'GZMB',
 'PAK2',
 'ARHGAP10',
 'PSMD8',
 'PSMD6',
 'SEM1',
 'PSMD3',
 'PSMD11',
 'PSMD14',
 'PSMD7',
 'PSMD12',
 'PSMD13',
 'PSMD1',
 'ADRM1',
 'PSMC3',
 'PSMC6',
 'PSMC4',
 'PSMC5',
 'PSMC1',
 'PSMC2',
 'PSMD2',
 'PSMA7',
 'PSMA1',
 